In [13]:
# Imports
import pandas as pd
from random import randint
from src import *
from src.simulator import SIMULATOR

In [14]:
sim = SIMULATOR()

# --------------------------------------------
#               KERNEL CONFIGURATION
# --------------------------------------------
kernel_path = './kernels/mmul/'
kernel_number = 1 
column_usage = [True, False] 
nInstrPerCol = 47 
imem_add_start = 0 
srf_spm_addres = 0 
version="_bb16x16_1col"

sim.kernel_config(column_usage, nInstrPerCol, imem_add_start, srf_spm_addres, kernel_number)

In [15]:
# --------------------------------------------
#                DATA SIZES
# --------------------------------------------
# DISCO-CGRA Configuration
nRCs = 4
nElementsPerVWRSlice = 32
nColsCGRA = 2

# Basic Block
BB_ROWS_A = 8
BB_COLS_A = 16
BB_ROWS_B = 16
BB_COLS_B = 8
# C 16x8 to fit VWR
BB_ROWS_C = 16
BB_COLS_C = 8

In [16]:
# Our test
nRowsA = 16
nColsA = 16
nRowsB = nColsA
nColsB = 8

#matrix_A = np.random.randint(1, 15, size=(nRowsA*nColsA))
#matrix_B = np.random.randint(1, 15, size=(nColsA*nColsB))
matrix_A = np.array([i for i in range(nRowsA*nColsA)])
matrix_B = np.array([i for i in range(nRowsB*nColsB)])
matrix_C = np.zeros((nRowsA*nColsB), dtype=int)

In [ ]:
# --------------------------------------------
#                LOAD SPM DATA
# --------------------------------------------
# SPM[0] = SRF
# SPM[1] = A00, SPM[2] = A10
# SPM[3] = B00, SPM[4] = B01
# SPM[5] = C00, SPM[6] = C10
# --------------------------------------------
# SRF[0] = nItLoop1 = 16 = n elems of the same row of A per RC 
# SRF[1] = 
# SRF[2] = Line of the SPM where the block of A is stored (= 1)
# SRF[3] = Line of the SPM where the block of B is stored (= 3)
# SRF[4] = Line of the SPM where the block of C is stored (= 5)
# --------------------------------------------

# Default SPM lines
srf_spm_line = 0
a_spm_line = 1
b_spm_line = 3
c_spm_line = 5


# Default SRF values
srf = [0 for i in range(SPM_NWORDS)]
srf[0] = 16     # nIt (same for both cols)
srf[1] = 1      # SPM for A (same for both cols)
srf[2] = 3      # SPM for B (col 0)
srf[3] = 5      # SPM for C (col 0)
srf[4] = 4      # SPM for B (col 1) = col 0 + 1
srf[5] = 6      # SPM for C (col 1) = col 0 + 1   
sim.setSPMLine(srf_spm_line, srf.copy())

# Prepare A
sim.setSPMLine(a_spm_line, matrix_A[:N_ELEMS_PER_VWR])
a_spm_line+=1
sim.setSPMLine(a_spm_line, matrix_A[N_ELEMS_PER_VWR:])

# Prepare B
matrix_B_reshaped = matrix_B.reshape(nColsA, nColsB) # Reshape into a 2D matrix
matrix_B_transposed = matrix_B_reshaped.T.flatten() # Transpose and flatten back into a 1D array
sim.setSPMLine(b_spm_line, matrix_B_transposed)

# Prepare C
sim.setSPMLine(c_spm_line, matrix_C)

sim.displaySPMLine(0)
sim.displaySPMLine(1)
sim.displaySPMLine(2)
sim.displaySPMLine(3)

In [ ]:
# --------------------------------------------
#              COMPILE ASM TO HEX
# --------------------------------------------
sim.compileAsmToHex(kernel_path, kernel_number, version=version)

Finally, we load the kernel into the internal memory of the specialized units and run it.

In [ ]:
# --------------------------------------------
#                 LOAD KERNEL
# --------------------------------------------

# This needs the hex instructions, if you don't provide them, generate then compiling the asm
sim.kernel_load(kernel_path, version=version + "_autogen", kernel_number=kernel_number)

# --------------------------------------------
#               SIMULATE EXECUTION
# --------------------------------------------
show_lcu = []
show_srf = []
show_lsu = []
show_rcs = [[],[],[],[]]
show_mxcu = []
display_ops = [show_lcu, show_lsu, show_mxcu, show_rcs, show_srf]

sim.run(kernel_number, display_ops=display_ops, max_iter=3000)

We can check it more rigorously. We can define our function in python and check that the output matches the CGRA output.

In [20]:
def mmul (in_A, in_B, nRowsA, nColsA, nColsB):
    out = np.zeros(nRowsA*nColsB)
    for i in range(nRowsA):
        for j in range(nColsB):
            sum = 0
            for k in range(nColsA):
                sum += int(in_A[i*nColsA + k] * in_B[k*nColsB + j])
            out[i*nColsB + j] = sum
    return [int(elem) for elem in out]

In [21]:
# Get output from the CGRA
disco_cgra_res = sim.getSPMLine(c_spm_line)

In [22]:
from itertools import groupby

def comprimir_rangos(arr):
    arr.sort()  # Asegurarse de que esté ordenado
    rangos = []
    
    for _, grupo in groupby(enumerate(arr), lambda x: x[1] - x[0]):
        grupo = [x[1] for x in grupo]  # Extraer los valores
        if len(grupo) > 1:
            rangos.append(f"{grupo[0]}-{grupo[-1]}")
        else:
            rangos.append(f"{grupo[0]}")

    return ", ".join(rangos)

def imprimir_por_linea(arr, tam_linea=8):
    for i in range(0, len(arr), tam_linea):
        print([int(x) for x in arr[i:i+tam_linea]])

In [ ]:
# Prepare output
# Extraer bloques C0, C1, ..., C7 en orden correcto
nBloques = 16
tamBloque = 8
# Reorganizar los bloques en el orden correcto
array_ordenado = []
for i in range(nColsCGRA):
    ini = 2*tamBloque*i
    for j in range (nRCs):
        array_ordenado.extend(disco_cgra_res[ini:ini+2*tamBloque])
        ini += 4*tamBloque

In [ ]:
errors_idx = []
expected_output = mmul(matrix_A, matrix_B, nRowsA, nColsA, nColsB)
for i in range(len(expected_output)):
    if expected_output[i] != array_ordenado[i]:
        errors_idx.append(i)
if len(errors_idx) == 0:
    print("The result is correct!")
else:
    print("Oops, something went wrong. There are " + str(len(errors_idx)) + " errors.")
    print(comprimir_rangos(errors_idx))
    print("DISCO-CGRA result:")
    imprimir_por_linea(disco_cgra_res)
    print("DISCO-CGRA reordered:")
    imprimir_por_linea(array_ordenado)
    print("Expected result:")
    imprimir_por_linea(expected_output)